# Active Learning with Uncertainty

Use model uncertainty to select informative training data.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from deepuq.active import ActiveLearningLoop, UncertaintySampling, RandomSampling
from deepuq.models import MLP
from deepuq.methods.mc_dropout import MCDropoutWrapper

## Setup

In [ ]:
# Generate 1D function with complex structure
np.random.seed(42)
torch.manual_seed(42)

def target_function(x):
    return np.sin(3*x) * np.exp(-0.3*x) + 0.5 * np.cos(5*x)

# Pool of unlabeled points
X_pool = np.linspace(0, 6, 200).reshape(-1, 1)
y_pool = target_function(X_pool) + np.random.randn(*X_pool.shape) * 0.05

# Initial training set: 5 random points
init_idx = np.random.choice(200, 5, replace=False)
X_init = X_pool[init_idx]
y_init = y_pool[init_idx]

print(f"Initial training points: {len(X_init)}")
print(f"Pool size: {len(X_pool)}")

plt.figure(figsize=(10, 4))
plt.plot(X_pool.ravel(), target_function(X_pool).ravel(), "k--", label="True function")
plt.scatter(X_init.ravel(), y_init.ravel(), c="red", s=100, zorder=5, label="Initial data")
plt.legend()
plt.title("Target Function and Initial Data")
plt.show()

## Run Active Learning Loop

In [ ]:
# Create model with MC Dropout for uncertainty
model = MLP(input_dim=1, hidden_dims=[64, 64], output_dim=1, dropout=0.1)
uq_model = MCDropoutWrapper(model, n_samples=30)

# Uncertainty sampling strategy
strategy = UncertaintySampling()

# Run active learning loop
loop = ActiveLearningLoop(
    model=uq_model,
    strategy=strategy,
    X_pool=torch.tensor(X_pool, dtype=torch.float32),
    y_pool=torch.tensor(y_pool, dtype=torch.float32),
    X_init=torch.tensor(X_init, dtype=torch.float32),
    y_init=torch.tensor(y_init, dtype=torch.float32),
    n_iterations=10,
    batch_size=5,
    train_epochs=100,
)

history = loop.run()
print(f"Final training set size: {history.n_acquired + len(X_init)}")

## Visualize Acquisition

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))

# True function
X_dense = np.linspace(0, 6, 500).reshape(-1, 1)
ax.plot(X_dense.ravel(), target_function(X_dense).ravel(), "k--", linewidth=2, label="True function")

# Final model predictions
X_test_t = torch.tensor(X_dense, dtype=torch.float32)
mean, std = uq_model.predict_uq(X_test_t)
mean, std = mean.numpy(), std.numpy()
ax.fill_between(X_dense.ravel(), (mean - 2*std).ravel(), (mean + 2*std).ravel(),
                alpha=0.2, color="blue", label="± 2σ")
ax.plot(X_dense.ravel(), mean.ravel(), "b-", label="Model mean")

# Acquired points colored by order
acquired_X = history.acquired_X.numpy()
order = np.arange(len(acquired_X))
scatter = ax.scatter(acquired_X.ravel(), target_function(acquired_X).ravel(),
                     c=order, cmap="plasma", s=60, zorder=5, label="Acquired points")
plt.colorbar(scatter, ax=ax, label="Acquisition order")

# Initial points
ax.scatter(X_init.ravel(), y_init.ravel(), c="red", s=100, marker="^", zorder=6, label="Initial data")

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Active Learning: Uncertainty-Guided Acquisition")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## Compare Strategies

In [ ]:
# Compare random vs uncertainty sampling
results = {}

for name, strat in [("Uncertainty", UncertaintySampling()), ("Random", RandomSampling())]:
    model = MLP(input_dim=1, hidden_dims=[64, 64], output_dim=1, dropout=0.1)
    uq_model = MCDropoutWrapper(model, n_samples=30)
    
    loop = ActiveLearningLoop(
        model=uq_model,
        strategy=strat,
        X_pool=torch.tensor(X_pool, dtype=torch.float32),
        y_pool=torch.tensor(y_pool, dtype=torch.float32),
        X_init=torch.tensor(X_init, dtype=torch.float32),
        y_init=torch.tensor(y_init, dtype=torch.float32),
        n_iterations=10,
        batch_size=5,
        train_epochs=100,
    )
    history = loop.run()
    results[name] = history.mse_curve

# Plot learning curves
plt.figure(figsize=(8, 5))
for name, mse in results.items():
    plt.plot(range(len(mse)), mse, "-o", label=name)

plt.xlabel("Iteration")
plt.ylabel("MSE")
plt.title("Active Learning: Random vs Uncertainty Sampling")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()